# Augmentation Review Dashboard

A scrollable, web-style dashboard for **paleontology reviewers** to validate the
generated augmentations and their masks. For each source image it shows a
collapsible section; each augmentation is one aligned row:

| Original image | Original mask | Augmented image | Augmented mask | Metadata |

The metadata panel lists the **augmentation operations applied** (from the
augmentation pipeline) and **per-class statistics** for the augmented sample
(images-containing, instance counts via connected components, pixels, % pixels —
reusing the instance-counting logic from `6_3_Masks_visualizer.ipynb`).

**Hovering** over the original image or original mask in any row pops up that
source image's own per-class statistics table.

The dashboard renders inline **and** is saved as a standalone, self-contained
`.html` file you can download and share with reviewers who don't use Colab.


In [ ]:
# 1) Install dependencies (transformers/tqdm needed to import the pipeline module)
!pip -q install --upgrade transformers tqdm

In [ ]:
# 2) Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('Not in Colab / mount skipped:', e)

In [ ]:
# 3) Configure repo + dataset paths (robust to a dropped Drive mount)
import os, sys
from pathlib import Path

# If the Drive mount dropped, Path.cwd() itself raises OSError 107 — guard it.
try:
    _cwd = Path.cwd()
except OSError:
    os.chdir('/content'); _cwd = Path('/content')
    print('WARNING: working dir was on a dropped Drive mount. Remount Drive '
          '(Runtime > Restart session, then re-run cell 2) before continuing.')

REPO_ROOT = Path("/content/drive/My Drive/Payne_lab_swin_transformer")
if not REPO_ROOT.is_dir():
    REPO_ROOT = _cwd
    while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "code").is_dir():
        REPO_ROOT = REPO_ROOT.parent

PIPELINE_DIR = REPO_ROOT / "code" / "model_training_pipeline"
sys.path.insert(0, str(PIPELINE_DIR))
os.chdir(PIPELINE_DIR)
print("REPO_ROOT =", REPO_ROOT)

IMG_DIR = Path("/content/drive/My Drive/Petrographic images_ML work/labelled images_PS/ALL_LABELS/img")
MASK_DIR = Path("/content/drive/My Drive/Petrographic images_ML work/labelled images_PS/ALL_LABELS/masks_machine")
if not IMG_DIR.is_dir() or not MASK_DIR.is_dir():
    IMG_DIR = REPO_ROOT / "data" / "carbonate_imgs_and_masks" / "img"
    MASK_DIR = REPO_ROOT / "data" / "carbonate_imgs_and_masks" / "masks"
    print("Using repo sample data.")
print("IMG_DIR  =", IMG_DIR)
print("MASK_DIR =", MASK_DIR)

In [ ]:
# 4) Update the repo clone so the pipeline + augmentation modules are current
import subprocess

def _git(*a):
    r = subprocess.run(["git", "-C", str(REPO_ROOT), *a], capture_output=True, text=True)
    return (r.stdout + r.stderr).strip()

print("branch:", _git("rev-parse", "--abbrev-ref", "HEAD"))
print(_git("fetch", "origin"))
print(_git("pull", "--ff-only") or "(nothing to pull / not a fast-forward)")

In [ ]:
# 5) Imports — reuse the augmentation library + pipeline constants/dataset
import numpy as np
import torch
from tqdm.auto import tqdm

from ppl_augment import augment, read_raw_image, read_raw_mask, DEFAULT_AUG
from swin_training_pipeline_221 import (
    CarbonateSegmentationDataset, CLASS_NAMES, NUM_CLASSES, IGNORE_INDEX, colorize_mask,
)
if NUM_CLASSES != 18:
    raise RuntimeError(
        f"Stale pipeline module: NUM_CLASSES={NUM_CLASSES} (expected 18). Update the repo "
        "clone (cell 4 / checkout a branch with the 18-class reconciliation) and re-run."
    )

ds = CarbonateSegmentationDataset(
    root=str(IMG_DIR.parent), img_dir=str(IMG_DIR), mask_dir=str(MASK_DIR),
    transforms=None, normalize=False, strict=True, ignore_class_ids=(),
)
print("paired samples:", len(ds.pairs))

In [ ]:
# 6) Per-class statistics for a single mask
# Adapted from 6_3_Masks_visualizer.ipynb: connected-component instance counting.
from scipy import ndimage

CONNECTIVITY = 8       # 8 = diagonals connected; 4 = edge neighbours only
MIN_AREA = 0           # ignore blobs smaller than this many pixels (0 = keep all)
BACKGROUND_IDS = {0}   # classes that get no instance count
_struct = ndimage.generate_binary_structure(2, 2 if CONNECTIVITY == 8 else 1)

def class_stats(label2d):
    # Returns list of (id, name, instances|None, pixels, pct) for classes present
    # in this single mask (ignore index and out-of-range ids skipped).
    label2d = np.asarray(label2d)
    ids, counts = np.unique(label2d, return_counts=True)
    total = int(label2d.size)
    rows = []
    for _id, cnt in zip(ids, counts):
        _id = int(_id)
        if _id == IGNORE_INDEX or _id >= NUM_CLASSES or _id < 0:
            continue
        if _id in BACKGROUND_IDS:
            inst = None
        else:
            comp, n = ndimage.label(label2d == _id, structure=_struct)
            if MIN_AREA > 0 and n > 0:
                sizes = np.bincount(comp.ravel())[1:]
                n = int((sizes >= MIN_AREA).sum())
            inst = int(n)
        rows.append((_id, CLASS_NAMES[_id], inst, int(cnt), 100.0 * cnt / total))
    return rows

In [ ]:
# 7) HTML dashboard builders (self-contained: base64-embedded images)
import base64, io
import html as _html
from PIL import Image
from IPython.display import HTML, display

def make_palette(n):
    rng = np.random.default_rng(7)
    pal = [(int(rng.integers(40, 235)), int(rng.integers(40, 235)), int(rng.integers(40, 235))) for _ in range(n)]
    pal[0] = (35, 35, 35)  # background = dark grey
    return pal

PALETTE = make_palette(NUM_CLASSES)

def to_hwc(img_u8):
    return img_u8.permute(1, 2, 0).cpu().numpy().astype(np.uint8)

def _data_uri(im, fmt, quality=85):
    buf = io.BytesIO()
    if fmt == "JPEG":
        im.convert("RGB").save(buf, format="JPEG", quality=quality)
        mime = "jpeg"
    else:
        im.save(buf, format="PNG")
        mime = "png"
    return "data:image/" + mime + ";base64," + base64.b64encode(buf.getvalue()).decode()

def img_tag_photo(arr_hwc, max_px):
    im = Image.fromarray(arr_hwc)
    if max(im.size) > max_px:
        im.thumbnail((max_px, max_px))
    return "<img loading='lazy' src='" + _data_uri(im, "JPEG") + "'/>"

def img_tag_mask(label2d, max_px):
    col = np.array(colorize_mask(np.asarray(label2d).astype(np.uint8), PALETTE))
    im = Image.fromarray(col)
    if max(im.size) > max_px:
        im.thumbnail((max_px, max_px), Image.NEAREST)
    return "<img loading='lazy' src='" + _data_uri(im, "PNG") + "'/>"

def _stats_table(stats_rows):
    body = ""
    for cid, name, inst, px, pct in stats_rows:
        inst_s = "&mdash;" if inst is None else str(inst)
        body += ("<tr><td class='name'>" + _html.escape(name) + "</td>"
                 "<td>1</td><td>" + inst_s + "</td>"
                 "<td>" + format(px, ",") + "</td><td>" + format(pct, ".2f") + "%</td></tr>")
    return ("<table><tr><th class='name'>class</th><th>imgs</th><th>inst</th>"
            "<th>pixels</th><th>%</th></tr>" + body + "</table>")

def meta_html(ops, stats_rows):
    ops_items = "".join("<li>" + _html.escape(o) + "</li>" for o in ops)
    return ("<div class='meta'><div class='ops'><b>Augmentations applied</b><ul>"
            + ops_items + "</ul></div>" + _stats_table(stats_rows) + "</div>")

def _hover_cell(caption, img_tag, tooltip_html):
    return ("<div class='cell'><div class='cap'>" + _html.escape(caption) + "</div>"
            "<div class='hoverwrap'>" + img_tag
            + "<div class='tooltip'>" + tooltip_html + "</div></div></div>")

def _plain_cell(caption, img_tag):
    return ("<div class='cell'><div class='cap'>" + _html.escape(caption) + "</div>" + img_tag + "</div>")

def row_html(o_img, o_mask, o_img_name, o_mask_name, a_img, a_mask, meta, orig_stats_html):
    tip = "<b>Original-image class stats</b>" + orig_stats_html
    return ("<div class='row'>"
            + _hover_cell(o_img_name, o_img, tip)
            + _hover_cell(o_mask_name, o_mask, tip)
            + _plain_cell("augmented image", a_img)
            + _plain_cell("augmented mask", a_mask)
            + "<div class='cell'>" + meta + "</div></div>")

def section_html(title, rows_html, open_default=True):
    hdr = ("<div class='row colhdr'><div>Original image</div><div>Original mask</div>"
           "<div>Augmented image</div><div>Augmented mask</div><div>Metadata</div></div>")
    op = " open" if open_default else ""
    return ("<details class='orig-section'" + op + "><summary>" + _html.escape(title) + "</summary>"
            + hdr + "".join(rows_html) + "</details>")

_CSS_RULES = [
    ".dash { font-family: system-ui, -apple-system, sans-serif; color:#222; max-width:1500px; margin:0 auto; }",
    ".dash h1 { font-size:22px; }",
    ".orig-section { border:1px solid #ccc; border-radius:8px; margin:14px 0; }",
    ".orig-section > summary { font-size:17px; font-weight:600; cursor:pointer; padding:10px 12px;"
    " background:#eef1f5; border-radius:8px; position:sticky; top:0; }",
    ".row { display:grid; grid-template-columns: 1.1fr 1.1fr 1fr 1fr 1.5fr; gap:10px;"
    " align-items:start; padding:12px; border-top:1px solid #eee; }",
    ".colhdr { font-weight:600; font-size:13px; color:#444; background:#fafafa; }",
    ".cell { text-align:center; }",
    ".cell .cap { font-size:11px; color:#666; margin-bottom:4px; word-break:break-all; }",
    ".cell img { max-width:100%; height:auto; border:1px solid #ddd; border-radius:4px; }",
    ".meta { text-align:left; font-size:12px; }",
    ".meta .ops { background:#f5f7fa; border-radius:4px; padding:6px 8px; margin-bottom:6px; }",
    ".meta .ops ul { margin:4px 0 0 16px; padding:0; }",
    ".meta table, .tooltip table { border-collapse:collapse; width:100%; font-size:11px; }",
    ".meta th, .meta td, .tooltip th, .tooltip td { border:1px solid #e0e0e0; padding:2px 5px; text-align:right; }",
    ".meta td.name, .meta th.name, .tooltip td.name, .tooltip th.name { text-align:left; }",
    ".hoverwrap { position:relative; display:block; }",
    ".hoverwrap .tooltip { display:none; position:absolute; z-index:30; top:0; left:103%;"
    " width:270px; background:#fff; border:1px solid #888; border-radius:6px;"
    " box-shadow:0 6px 18px rgba(0,0,0,.28); padding:8px; text-align:left; }",
    ".hoverwrap:hover .tooltip { display:block; }",
    ".tooltip b { font-size:12px; display:block; margin-bottom:4px; }",
]
CSS = "<style>" + "".join(_CSS_RULES) + "</style>"

def page_html(sections, header):
    return "<div class='dash'>" + CSS + "<h1>" + _html.escape(header) + "</h1>" + "".join(sections) + "</div>"

## Build & display the dashboard

Adjust `N_ORIGINALS` / `N_AUG` below. For very large datasets, render in batches
(e.g. set `START_IDX`) to keep the page light — each section is collapsible and
images load lazily.


In [ ]:
# 9) Generate the dashboard
N_ORIGINALS = 5      # number of source images to include
N_AUG = 5            # augmentations per source image
MAX_PX = 360         # on-screen size (px) of each embedded image
START_IDX = 0        # first ds.pairs index (use to render later batches)
SEED = 0             # set None for fresh randomness each run
OUT_HTML = "/content/augmentation_review_dashboard.html"

if SEED is not None:
    import random
    random.seed(SEED); torch.manual_seed(SEED)

end = min(START_IDX + N_ORIGINALS, len(ds.pairs))
sections = []
for i in tqdm(range(START_IDX, end), desc="originals"):
    img_path, mask_path = ds.pairs[i]
    img0 = read_raw_image(img_path)
    mask0 = read_raw_mask(mask_path)
    o_img = img_tag_photo(to_hwc(img0), MAX_PX)
    o_mask = img_tag_mask(mask0.numpy(), MAX_PX)
    orig_stats_html = _stats_table(class_stats(mask0.numpy()))  # once per source image (hover tooltip)
    rows = []
    for k in range(N_AUG):
        a_img, a_mask, ops = augment(img0.clone(), mask0.clone(), return_ops=True)
        meta = meta_html(ops, class_stats(a_mask.numpy()))
        rows.append(row_html(o_img, o_mask, img_path.name, mask_path.name,
                             img_tag_photo(to_hwc(a_img), MAX_PX),
                             img_tag_mask(a_mask.numpy(), MAX_PX), meta, orig_stats_html))
    sections.append(section_html(img_path.name + "  -  " + str(N_AUG) + " augmentations", rows))

page = page_html(sections, "Augmentation Review Dashboard (" + str(end - START_IDX) + " images x " + str(N_AUG) + " augs)")
with open(OUT_HTML, "w") as f:
    f.write(page)
print("Saved standalone dashboard ->", OUT_HTML)
print("Download via the Colab Files pane, or:  from google.colab import files; files.download(OUT_HTML)")
display(HTML(page))

## 12. Export the augmented 512x512 crops to Drive

Saves `N_AUG` augmentations per source image as image+mask pairs into
`augmented_data/img` and `augmented_data/masks_machine` on Drive. These are the
512x512 crops the model actually trains on (the same `augment()` pipeline shown in
the dashboard). With `SEED = 0` they're identical to the dashboard's samples; set
`SEED = None` for a fresh draw. Run after the setup cells (through the dataset cell).


In [ ]:
# === Export 512x512 augmented crops (image + mask) to Drive ===
import random, torch
from pathlib import Path
from tqdm.auto import tqdm
from torchvision.io import write_jpeg, write_png
from ppl_augment import augment, read_raw_image, read_raw_mask

# ---- config ----
N_AUG = 10                # augmentations per source image (89 x 10 = 890)
SEED  = 0                 # set to None for different augmentations on a re-run
DEST  = Path("/content/drive/My Drive/Fusion AI/Payne project/Petrographic images_ML work/augmented_data")
IMG_OUT, MASK_OUT = DEST / "img", DEST / "masks_machine"
IMG_OUT.mkdir(parents=True, exist_ok=True)
MASK_OUT.mkdir(parents=True, exist_ok=True)

if SEED is not None:
    random.seed(SEED); torch.manual_seed(SEED)

written = 0
for img_path, mask_path in tqdm(ds.pairs, desc="source images"):
    img0, mask0 = read_raw_image(img_path), read_raw_mask(mask_path)
    for k in range(N_AUG):
        a_img, a_mask = augment(img0.clone(), mask0.clone())     # 512x512 crop + augmentations
        stem = f"{img_path.stem}_aug{k:02d}"
        write_jpeg(a_img.contiguous(), str(IMG_OUT / f"{stem}.jpg"), quality=95)
        write_png(a_mask.to(torch.uint8).unsqueeze(0).contiguous(), str(MASK_OUT / f"{stem}.png"))
        written += 1

print(f"\nWrote {written} augmented 512x512 pairs:\n  images -> {IMG_OUT}\n  masks  -> {MASK_OUT}")